In [1]:
import QuantLib as ql
import pandas as pd

today = ql.Date(25, ql.October, 2021)
ql.Settings.instance().evaluationDate = today

The process of bootstrapping, unlike fitting, uses an iterative algorithm to create a curve that reprices a number of instruments exactly. Each instrument, starting with the one with the shortest maturity, adds a node to the curve keeping the previous ones unchanged until the full curve is built.

When instruments need different curves, we can build them one by one, usually starting from the curve corresponding to the overnight index, e.g., SOFR in a US setting or ESTR in the Euro zone; this is because overnight-based instrument (usually futures and overnight-indexed swaps) only need that one curve, while other instruments need the overnight curve for discounting as well as other curves for forecasting.

<h2>Risk-free curves</h2>

These curves are used to forecast risk-free rates such as SOFR(Secured Overnight Financing Rate), SONIA(Sterling Overnight Index Average) or ESTR(Euro Short Term Rate), as well as for risk-free discounting. In this case, I’ll use SOFR; of course, we have a class for that.

In [2]:
index = ql.Sofr()

As I mentioned, the curve is built based on different kinds of instruments. The library defines helper classes corresponding to each of them, each instance acting as an objective function for the solver underlying the bootstrapping process.

For futures, the quote to reproduce is a price…

In [4]:
futures_data = [
    ((11, 2021, ql.Monthly), 99.934),
    ((12, 2021, ql.Monthly), 99.922),
    ((1, 2022, ql.Monthly), 99.914),
    ((2, 2022, ql.Monthly), 99.919),
    ((3, 2022, ql.Quarterly), 99.876),
    ((6, 2022, ql.Quarterly), 99.799),
    ((9, 2022, ql.Quarterly), 99.626),
    ((12, 2022, ql.Quarterly), 99.443),
]

…while for OIS(Overnight Index Swap - fixed interest rate vs overnight floating rate), the quote is their fixed rate. In the list below, that shows quotes as they might come from some provider, the rates are in percent unit (that is, the 10-years rate is 1.359%); the library will require them in decimal units (i.e., 0.01359 for the same rate.)

In [5]:
ois_data = [
    ("1Y", 0.137),
    ("2Y", 0.409),
    ("3Y", 0.674),
    ("5Y", 1.004),
    ("8Y", 1.258),
    ("10Y", 1.359),
    ("12Y", 1.420),
    ("15Y", 1.509),
    ("20Y", 1.574),
    ("25Y", 1.586),
    ("30Y", 1.579),
    ("35Y", 1.559),
    ("40Y", 1.514),
    ("45Y", 1.446),
    ("50Y", 1.425),
]

Quotes are usually wrapped in SimpleQuote instances, so that they can be changed later; when this happens, the curve will then be notified and recalculate when needed. Once wrapped, they are passed to the helpers.

In [6]:
futures_helpers = []

for (month, year, frequency), price in futures_data:
    q = ql.SimpleQuote(price)
    futures_helpers.append(
        ql.SofrFutureRateHelper(ql.QuoteHandle(q), month, year, frequency)
    )

In [7]:
settlement_days = 2

ois_quotes = {}
ois_helpers = []

for tenor, quote in ois_data:
    q = ql.SimpleQuote(quote/100.0)
    ois_quotes[tenor] = q
    ois_helpers.append(
        ql.OISRateHelper(
            settlement_days,
            ql.Period(tenor),
            ql.QuoteHandle(q),
            index,
            paymentFrequency = ql.Annual
        )
    )


Another thing to note is that the ranges of futures and OIS can overlap; for instance, in this case, the maturity of the last futures is later than the maturity of the first OIS.

In [8]:
futures_helpers[-1].latestDate()

Date(15,3,2023)

In [9]:
ois_helpers[0].latestDate()

Date(27,10,2022)

In a real set of data, you would also have OIS quotes for maturities below the year. Using all the available helpers might cause collisions between maturities (which, in turn, would cause the bootstrap to fail); and even if that doesn’t happen, having nodes too close might result in unrealistic forwards between them. You’ll have to decide what to do: one possibility is to use only futures in the short range (since they’re usually more liquid) and start using swaps after the maturity of the last futures, possibly with a bit of buffer space, as in the cell below:

In [11]:
cutoff_date = futures_helpers[-1].latestDate() + ql.Period(1, ql.Months)
ois_helpers = [h for h in ois_helpers if h.latestDate() > cutoff_date]

In [14]:
sofr_curve = ql.PiecewiseLogCubicDiscount(
    0,
    ql.UnitedStates(ql.UnitedStates.GovernmentBond),
    futures_helpers + ois_helpers,
    ql.Actual360(),
)

Depending on the chosen class, the nodes between which the curve interpolates will contain discount factors (as in this case), zero rates, or instantaneous forward rates.

In [15]:
pd.DataFrame(sofr_curve.nodes(),columns=["date","discount factor"])

,date,discount factor
0,"October 25th, 2021",1.000000
1,"December 1st, 2021",0.999933
2,"January 1st, 2022",0.999866
3,"February 1st, 2022",0.999792
4,"March 1st, 2022",0.999729
5,"June 15th, 2022",0.999378
6,"September 21st, 2022",0.998831
7,"December 21st, 2022",0.997888
8,"March 15th, 2023",0.996593
9,"October 27th, 2023",0.991743


And just for kicks, we can visualize the curve we just created:

In [ ]:
from matplotlib import pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np

